In [1]:
# ============================================
# CELL 1 ? IMPORTS + PATHS
# ============================================
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import ElasticNet, Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error

STAGE0_DIR = Path("/kaggle/input/datasets/nlvdull/stage-0-update/stage0_artifacts")
STAGE4_INPUT_DIR = Path("/kaggle/input/datasets/nlvdull/stage-3-elec/stage3_artifacts")
STAGE4_DIR = Path("/kaggle/working/stage4_artifacts")
STAGE4_DIR.mkdir(parents=True, exist_ok=True)

COL_USER   = "reviewerID"
COL_ITEM   = "asin"
COL_RATING = "overall"
COL_DATE   = "review_date"

print("Stage0 :", STAGE0_DIR)
print("Stage4 input :", STAGE4_INPUT_DIR)
print("Stage4 output:", STAGE4_DIR)


Stage0 : /kaggle/input/datasets/nlvdull/stage-0-update/stage0_artifacts
Stage4 input : /kaggle/input/datasets/nlvdull/stage-3-elec/stage3_artifacts
Stage4 output: /kaggle/working/stage4_artifacts


In [ ]:
# ============================================
# CELL 2 ? LOAD STAGE 0 + BUILD TEST_WARM
# ============================================
with open(STAGE0_DIR / "split_meta.json", "r") as f:
    split_meta = json.load(f)

with open(STAGE0_DIR / "baseline_scores.json", "r") as f:
    baseline_scores = json.load(f)

train = pd.read_parquet(STAGE0_DIR / "train.parquet").copy()
test  = pd.read_parquet(STAGE0_DIR / "test.parquet").copy()
user_means_df = pd.read_parquet(STAGE0_DIR / "user_means.parquet")
item_means_df = pd.read_parquet(STAGE0_DIR / "item_means.parquet")

if COL_DATE in train.columns:
    train[COL_DATE] = pd.to_datetime(train[COL_DATE], errors="coerce")
if COL_DATE in test.columns:
    test[COL_DATE] = pd.to_datetime(test[COL_DATE], errors="coerce")

if COL_USER in user_means_df.columns:
    user_mean_map = dict(zip(user_means_df[COL_USER], user_means_df["user_mean"]))
else:
    user_mean_map = user_means_df["user_mean"].to_dict()

if COL_ITEM in item_means_df.columns:
    item_mean_map = dict(zip(item_means_df[COL_ITEM], item_means_df["item_mean"]))
else:
    item_mean_map = item_means_df["item_mean"].to_dict()

GLOBAL_MEAN = float(train[COL_RATING].mean())
train_users = set(train[COL_USER].unique())
train_items = set(train[COL_ITEM].unique())

test = test.copy()
test["user_known"] = test[COL_USER].isin(train_users)
test["item_known"] = test[COL_ITEM].isin(train_items)
mask_warm = test["user_known"] & test["item_known"]
test_warm = test.loc[mask_warm].reset_index(drop=True)

def clip_rating(x):
    return np.clip(np.asarray(x, dtype=np.float32), 1.0, 5.0)

def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def mae(y_true, y_pred):
    return float(mean_absolute_error(y_true, y_pred))

def evaluate_predictions(y_true, y_pred, name="model"):
    y_true = np.asarray(y_true, dtype=np.float32)
    y_pred = clip_rating(y_pred)
    return {"model": name, "n": int(len(y_true)), "rmse": rmse(y_true, y_pred), "mae": mae(y_true, y_pred)}

def baseline_predict(df_subset, user_map, item_map, global_mean):
    u = df_subset[COL_USER].map(user_map).fillna(global_mean)
    i = df_subset[COL_ITEM].map(item_map).fillna(global_mean)
    return ((u + i) / 2.0).clip(1.0, 5.0).values.astype(np.float32)

def to_jsonable(obj):
    if isinstance(obj, dict):
        return {str(k): to_jsonable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [to_jsonable(v) for v in obj]
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, (np.bool_,)):
        return bool(obj)
    return obj

y_true = test_warm[COL_RATING].astype(np.float32).values
y_base = baseline_predict(test_warm, user_mean_map, item_mean_map, GLOBAL_MEAN)

print("Warm test size:", len(test_warm))
print(evaluate_predictions(y_true, y_base, "baseline"))


In [3]:
# ============================================
# CELL 3 — LOAD STAGE 3 MODEL OUTPUTS
# ============================================
y_a = np.load(STAGE4_INPUT_DIR / "model_a_preds.npy")
y_b = np.load(STAGE4_INPUT_DIR / "model_b_preds.npy")
y_c = np.load(STAGE4_INPUT_DIR / "model_c_preds.npy")
y_d = np.load(STAGE4_INPUT_DIR / "model_d_preds.npy")

print("Loaded predictions:")
print("A:", y_a.shape)
print("B:", y_b.shape)
print("C:", y_c.shape)
print("D:", y_d.shape)

assert len(y_a) == len(test_warm)
assert len(y_b) == len(test_warm)
assert len(y_c) == len(test_warm)
assert len(y_d) == len(test_warm)

Loaded predictions:
A: (905868,)
B: (905868,)
C: (905868,)
D: (905868,)


In [4]:
# ============================================
# CELL 4 — QUICK SINGLE-MODEL CHECK
# ============================================
results_single = [
    evaluate_predictions(y_true, y_base, "baseline"),
    evaluate_predictions(y_true, y_a, "model_a"),
    evaluate_predictions(y_true, y_b, "model_b"),
    evaluate_predictions(y_true, y_c, "model_c"),
    evaluate_predictions(y_true, y_d, "model_d"),
]

results_single_df = pd.DataFrame(results_single).sort_values("rmse").reset_index(drop=True)
display(results_single_df)

,model,n,rmse,mae
0,model_b,905868,1.212699,0.868733
1,model_c,905868,1.213060,0.924246
2,baseline,905868,1.213440,0.925114
3,model_d,905868,1.213454,0.924767
4,model_a,905868,1.215371,0.924901


In [ ]:
# ============================================
# CELL 5 ? ENSEMBLE HELPERS
# ============================================
import itertools
import warnings

warnings.filterwarnings("ignore", category=ConvergenceWarning)

ADVANCED_FAMILIES = {"meta_ridge", "meta_elasticnet", "meta_hist_gradient_boosting", "group_aware_ridge"}


def weighted_blend(preds_dict, weights_dict):
    total_w = 0.0
    out = np.zeros_like(next(iter(preds_dict.values())), dtype=np.float32)
    for key, pred in preds_dict.items():
        w = float(weights_dict.get(key, 0.0))
        if w == 0.0:
            continue
        out += w * np.asarray(pred, dtype=np.float32)
        total_w += w
    if total_w <= 0:
        raise ValueError("All weights are zero.")
    return clip_rating(out / total_w)


def build_candidate_record(model_name, family, oof_pred, test_pred, weights=None, params=None, eligible_for_selection=True):
    weights = {} if weights is None else weights
    params = {} if params is None else params
    oof_res = evaluate_predictions(y_val_true, oof_pred, model_name)
    test_res = evaluate_predictions(y_true, test_pred, model_name)
    return {
        "model": model_name,
        "family": family,
        "eligible_for_selection": bool(eligible_for_selection),
        "oof_rmse": float(oof_res["rmse"]),
        "oof_mae": float(oof_res["mae"]),
        "test_rmse": float(test_res["rmse"]),
        "test_mae": float(test_res["mae"]),
        "weights_json": json.dumps(to_jsonable(weights), sort_keys=True),
        "params_json": json.dumps(to_jsonable(params), sort_keys=True),
        "selection_source": "oof_validation",
    }


def register_candidate(model_name, family, oof_pred, test_pred, weights=None, params=None, eligible_for_selection=True):
    record = build_candidate_record(model_name, family, oof_pred, test_pred, weights, params, eligible_for_selection)
    compare_rows.append(record)
    candidate_store[model_name] = {
        "family": family,
        "oof_pred": clip_rating(np.asarray(oof_pred, dtype=np.float32)),
        "test_pred": clip_rating(np.asarray(test_pred, dtype=np.float32)),
        "weights": to_jsonable({} if weights is None else weights),
        "params": to_jsonable({} if params is None else params),
        "eligible_for_selection": bool(eligible_for_selection),
    }
    return record


def evaluate_per_star(y_true_arr, y_pred_arr):
    y_true_arr = np.asarray(y_true_arr, dtype=np.float32)
    y_pred_arr = clip_rating(y_pred_arr)
    star_values = np.rint(y_true_arr).astype(np.int32)
    out = {}
    for star in range(1, 6):
        mask = star_values == star
        out[f"star_{star}_n"] = int(mask.sum())
        if mask.sum() == 0:
            out[f"star_{star}_rmse"] = None
            out[f"star_{star}_mae"] = None
        else:
            out[f"star_{star}_rmse"] = rmse(y_true_arr[mask], y_pred_arr[mask])
            out[f"star_{star}_mae"] = mae(y_true_arr[mask], y_pred_arr[mask])
    return out


def simplex_weight_dicts(keys, step):
    total_units = int(round(1.0 / step))
    for units in itertools.product(range(total_units + 1), repeat=len(keys)):
        if sum(units) != total_units:
            continue
        yield {key: float(unit * step) for key, unit in zip(keys, units)}


def search_best_convex_blend(model_name, keys, step):
    best = None
    for weights in simplex_weight_dicts(keys, step):
        oof_pred = weighted_blend({key: pred_bank_val[key] for key in keys}, weights)
        test_pred = weighted_blend({key: pred_bank_test[key] for key in keys}, weights)
        record = build_candidate_record(model_name, "grid_blend", oof_pred, test_pred, weights, {"keys": list(keys), "step": float(step)}, eligible_for_selection=True)
        candidate = {
            "record": record,
            "oof_pred": oof_pred.copy(),
            "test_pred": test_pred.copy(),
            "weights": to_jsonable(weights),
            "params": {"keys": list(keys), "step": float(step)},
        }
        if best is None or (record["oof_rmse"], record["oof_mae"]) < (best["record"]["oof_rmse"], best["record"]["oof_mae"]):
            best = candidate
    return best


def search_best_residual_blend(step=0.10):
    base_val = pred_bank_val["base"]
    base_test = pred_bank_test["base"]
    best = None
    total_units = int(round(1.0 / step))
    for wb in range(total_units + 1):
        for wc in range(total_units + 1):
            for wd in range(total_units + 1):
                w_b = float(wb * step)
                w_c = float(wc * step)
                w_d = float(wd * step)
                oof_pred = clip_rating(base_val + w_b * (pred_bank_val["b"] - base_val) + w_c * (pred_bank_val["c"] - base_val) + w_d * (pred_bank_val["d"] - base_val))
                test_pred = clip_rating(base_test + w_b * (pred_bank_test["b"] - base_test) + w_c * (pred_bank_test["c"] - base_test) + w_d * (pred_bank_test["d"] - base_test))
                weights = {"base": 1.0, "w_b": w_b, "w_c": w_c, "w_d": w_d}
                params = {"step": float(step), "formula": "base + w_b*(B-base) + w_c*(C-base) + w_d*(D-base)"}
                record = build_candidate_record("residual_blend_base_b_c_d", "residual_blend", oof_pred, test_pred, weights, params, eligible_for_selection=True)
                candidate = {
                    "record": record,
                    "oof_pred": oof_pred.copy(),
                    "test_pred": test_pred.copy(),
                    "weights": to_jsonable(weights),
                    "params": params,
                }
                if best is None or (record["oof_rmse"], record["oof_mae"]) < (best["record"]["oof_rmse"], best["record"]["oof_mae"]):
                    best = candidate
    return best


def build_residual_meta_features(pred_base, pred_a, pred_b, pred_c, pred_d):
    pred_base = np.asarray(pred_base, dtype=np.float32)
    pred_a = np.asarray(pred_a, dtype=np.float32)
    pred_b = np.asarray(pred_b, dtype=np.float32)
    pred_c = np.asarray(pred_c, dtype=np.float32)
    pred_d = np.asarray(pred_d, dtype=np.float32)
    pred_stack = np.column_stack([pred_a, pred_b, pred_c, pred_d]).astype(np.float32)
    feature_names = [
        "b_minus_base",
        "c_minus_base",
        "d_minus_base",
        "a_minus_base",
        "abs_b_minus_base",
        "abs_c_minus_base",
        "abs_d_minus_base",
        "abs_a_minus_base",
        "prediction_mean",
        "prediction_std",
        "prediction_range",
        "base",
    ]
    X = np.column_stack([
        pred_b - pred_base,
        pred_c - pred_base,
        pred_d - pred_base,
        pred_a - pred_base,
        np.abs(pred_b - pred_base),
        np.abs(pred_c - pred_base),
        np.abs(pred_d - pred_base),
        np.abs(pred_a - pred_base),
        pred_stack.mean(axis=1),
        pred_stack.std(axis=1),
        pred_stack.max(axis=1) - pred_stack.min(axis=1),
        pred_base,
    ]).astype(np.float32)
    return X, feature_names


def build_group_labels(user_counts, item_counts):
    user_counts = np.asarray(user_counts, dtype=np.int32)
    item_counts = np.asarray(item_counts, dtype=np.int32)
    user_bin = np.where(user_counts <= 2, "low", np.where(user_counts <= 10, "mid", "high"))
    item_bin = np.where(item_counts <= 2, "low", np.where(item_counts <= 20, "mid", "high"))
    return np.char.add(np.char.add(user_bin.astype(str), "_"), item_bin.astype(str))


def fit_group_aware_ridge_candidate(X_val, X_test, y_val_target, fallback_val_pred, fallback_test_pred, val_groups, test_groups, min_rows=5000, alpha=1.0):
    X_val = np.asarray(X_val, dtype=np.float32)
    X_test = np.asarray(X_test, dtype=np.float32)
    y_val_target = np.asarray(y_val_target, dtype=np.float32)
    val_groups = np.asarray(val_groups)
    test_groups = np.asarray(test_groups)
    oof_pred = clip_rating(np.asarray(fallback_val_pred, dtype=np.float32).copy())
    test_pred = clip_rating(np.asarray(fallback_test_pred, dtype=np.float32).copy())
    group_summary = {}
    unique_groups = sorted(set(val_groups.tolist()) | set(test_groups.tolist()))
    for group in unique_groups:
        val_mask = val_groups == group
        test_mask = test_groups == group
        n_val = int(val_mask.sum())
        n_test = int(test_mask.sum())
        if n_val < min_rows:
            group_summary[group] = {"strategy": "fallback_global_ridge", "n_val": n_val, "n_test": n_test, "reason": f"n_val < {min_rows}"}
            continue
        try:
            model = Ridge(alpha=float(alpha), positive=True, fit_intercept=False)
            model.fit(X_val[val_mask], y_val_target[val_mask])
            oof_pred[val_mask] = clip_rating(model.predict(X_val[val_mask]))
            if n_test > 0:
                test_pred[test_mask] = clip_rating(model.predict(X_test[test_mask]))
            coef = model.coef_.astype(np.float32)
            group_summary[group] = {
                "strategy": "group_specific_positive_ridge",
                "n_val": n_val,
                "n_test": n_test,
                "alpha": float(alpha),
                "coef": {"base": float(coef[0]), "b": float(coef[1]), "c": float(coef[2]), "d": float(coef[3])},
            }
        except Exception as exc:
            group_summary[group] = {"strategy": "fallback_global_ridge", "n_val": n_val, "n_test": n_test, "reason": str(exc)}
    return clip_rating(oof_pred), clip_rating(test_pred), group_summary


In [ ]:
# ============================================
# CELL 6 ? LOAD STRICT OOF VALIDATION INPUTS
# ============================================
y_val_true = np.load(STAGE4_INPUT_DIR / "y_val_true_oof.npy")
y_val_base = np.load(STAGE4_INPUT_DIR / "baseline_val_preds_oof.npy")
y_val_a    = np.load(STAGE4_INPUT_DIR / "model_a_val_preds_oof.npy")
y_val_b    = np.load(STAGE4_INPUT_DIR / "model_b_val_preds_oof.npy")
y_val_c    = np.load(STAGE4_INPUT_DIR / "model_c_val_preds_oof.npy")
y_val_d    = np.load(STAGE4_INPUT_DIR / "model_d_val_preds_oof.npy")
validation_index_oof = pd.read_csv(STAGE4_INPUT_DIR / "validation_index_oof.csv")

for required_col in [COL_USER, COL_ITEM]:
    assert required_col in validation_index_oof.columns, f"Missing column in validation_index_oof.csv: {required_col}"

for arr_name, arr in {"baseline": y_val_base, "model_a": y_val_a, "model_b": y_val_b, "model_c": y_val_c, "model_d": y_val_d}.items():
    assert len(arr) == len(y_val_true), f"Length mismatch for {arr_name}: {len(arr)} vs {len(y_val_true)}"
assert len(validation_index_oof) == len(y_val_true), f"validation_index_oof length mismatch: {len(validation_index_oof)} vs {len(y_val_true)}"
for arr_name, arr in {"baseline": y_base, "model_a": y_a, "model_b": y_b, "model_c": y_c, "model_d": y_d}.items():
    assert len(arr) == len(y_true), f"Test prediction length mismatch for {arr_name}: {len(arr)} vs {len(y_true)}"

pred_bank_val = {"base": y_val_base, "a": y_val_a, "b": y_val_b, "c": y_val_c, "d": y_val_d}
pred_bank_test = {"base": y_base, "a": y_a, "b": y_b, "c": y_c, "d": y_d}

compare_rows = []
candidate_store = {}
advanced_diagnostics = {"candidate_search": {}, "skipped_candidates": {}, "meta_feature_names": []}

reference_candidates = {
    "baseline": ("single_model", y_val_base, y_base, {"base": 1.0}, {"kind": "single_model"}),
    "model_a": ("single_model", y_val_a, y_a, {"a": 1.0}, {"kind": "single_model"}),
    "model_b": ("single_model", y_val_b, y_b, {"b": 1.0}, {"kind": "single_model"}),
    "model_c": ("single_model", y_val_c, y_c, {"c": 1.0}, {"kind": "single_model"}),
    "model_d": ("single_model", y_val_d, y_d, {"d": 1.0}, {"kind": "single_model"}),
}
for name, (family, oof_pred, test_pred, weights, params) in reference_candidates.items():
    register_candidate(name, family, oof_pred, test_pred, weights=weights, params=params, eligible_for_selection=True)

reference_df = pd.DataFrame(compare_rows).sort_values(["oof_rmse", "oof_mae", "model"]).reset_index(drop=True)
display(reference_df)
print("Loaded strict OOF rows:", len(y_val_true))
print("Loaded validation_index_oof rows:", len(validation_index_oof))


In [ ]:
# ============================================
# CELL 7 ? MANUAL BLENDS + GRID BLENDS ON OOF
# ============================================
manual_configs = {
    "blend_base_b_manual": {"base": 0.50, "b": 0.50},
    "blend_base_b_c_manual": {"base": 0.55, "b": 0.35, "c": 0.10},
    "blend_base_b_c_d_manual": {"base": 0.55, "b": 0.30, "c": 0.10, "d": 0.05},
}

for name, weights in manual_configs.items():
    oof_pred = weighted_blend(pred_bank_val, weights)
    test_pred = weighted_blend(pred_bank_test, weights)
    register_candidate(name, "manual_blend", oof_pred, test_pred, weights=weights, params={"kind": "manual"}, eligible_for_selection=True)

grid_base_b = search_best_convex_blend("grid_base_b", ("base", "b"), step=0.05)
register_candidate("grid_base_b", "grid_blend", grid_base_b["oof_pred"], grid_base_b["test_pred"], weights=grid_base_b["weights"], params=grid_base_b["params"], eligible_for_selection=True)

grid_base_b_c = search_best_convex_blend("grid_base_b_c", ("base", "b", "c"), step=0.05)
register_candidate("grid_base_b_c", "grid_blend", grid_base_b_c["oof_pred"], grid_base_b_c["test_pred"], weights=grid_base_b_c["weights"], params=grid_base_b_c["params"], eligible_for_selection=True)

grid_base_b_c_d = search_best_convex_blend("grid_base_b_c_d", ("base", "b", "c", "d"), step=0.10)
register_candidate("grid_base_b_c_d", "grid_blend", grid_base_b_c_d["oof_pred"], grid_base_b_c_d["test_pred"], weights=grid_base_b_c_d["weights"], params=grid_base_b_c_d["params"], eligible_for_selection=True)

blend_preview_df = pd.DataFrame(compare_rows).sort_values(["oof_rmse", "oof_mae", "model"]).reset_index(drop=True)
display(blend_preview_df)


In [ ]:
# ============================================
# CELL 8 ? RESIDUAL BLEND + RIDGE + ADVANCED META
# ============================================
best_residual = search_best_residual_blend(step=0.10)
register_candidate("residual_blend_base_b_c_d", "residual_blend", best_residual["oof_pred"], best_residual["test_pred"], weights=best_residual["weights"], params=best_residual["params"], eligible_for_selection=True)

X_val_base_b_c_d = np.column_stack([y_val_base, y_val_b, y_val_c, y_val_d]).astype(np.float32)
X_test_base_b_c_d = np.column_stack([y_base, y_b, y_c, y_d]).astype(np.float32)

ridge = Ridge(alpha=1.0, positive=True, fit_intercept=False)
ridge.fit(X_val_base_b_c_d, y_val_true)
ridge_coef = ridge.coef_.astype(np.float32)
ridge_coef_sum = float(ridge_coef.sum())
ridge_coef_norm = (ridge_coef / ridge_coef_sum).astype(np.float32) if ridge_coef_sum > 0 else ridge_coef.copy()

y_pred_ridge_oof = clip_rating(ridge.predict(X_val_base_b_c_d))
y_pred_ridge = clip_rating(ridge.predict(X_test_base_b_c_d))
ridge_weights = {"base": float(ridge_coef[0]), "b": float(ridge_coef[1]), "c": float(ridge_coef[2]), "d": float(ridge_coef[3]), "normalized_base": float(ridge_coef_norm[0]), "normalized_b": float(ridge_coef_norm[1]), "normalized_c": float(ridge_coef_norm[2]), "normalized_d": float(ridge_coef_norm[3])}
ridge_params = {"alpha": 1.0, "positive": True, "fit_intercept": False}
register_candidate("ridge_positive_base_b_c_d", "ridge_positive", y_pred_ridge_oof, y_pred_ridge, weights=ridge_weights, params=ridge_params, eligible_for_selection=True)
advanced_diagnostics["candidate_search"]["ridge_positive_base_b_c_d"] = {"params": ridge_params, "weights": ridge_weights}

X_val_meta, meta_feature_names = build_residual_meta_features(y_val_base, y_val_a, y_val_b, y_val_c, y_val_d)
X_test_meta, _ = build_residual_meta_features(y_base, y_a, y_b, y_c, y_d)
y_res_val = (y_val_true - y_val_base).astype(np.float32)
advanced_diagnostics["meta_feature_names"] = meta_feature_names

ridge_meta_rows = []
best_ridge_meta = None
for alpha in [0.01, 0.1, 1.0, 5.0, 10.0]:
    model = Ridge(alpha=float(alpha), fit_intercept=True)
    model.fit(X_val_meta, y_res_val)
    oof_pred = clip_rating(y_val_base + model.predict(X_val_meta))
    test_pred = clip_rating(y_base + model.predict(X_test_meta))
    oof_res = evaluate_predictions(y_val_true, oof_pred, f"residual_ridge_meta_alpha_{alpha}")
    row = {"alpha": float(alpha), "oof_rmse": float(oof_res["rmse"]), "oof_mae": float(oof_res["mae"]), "intercept": float(model.intercept_), "coef_by_feature": {name: float(coef) for name, coef in zip(meta_feature_names, model.coef_)}}
    ridge_meta_rows.append(row)
    candidate = {"params": {"alpha": float(alpha), "fit_intercept": True, "feature_names": meta_feature_names}, "weights": {"intercept": float(model.intercept_), "coef_by_feature": {name: float(coef) for name, coef in zip(meta_feature_names, model.coef_)}}, "oof_pred": oof_pred.copy(), "test_pred": test_pred.copy(), "metrics": oof_res}
    if best_ridge_meta is None or (oof_res["rmse"], oof_res["mae"]) < (best_ridge_meta["metrics"]["rmse"], best_ridge_meta["metrics"]["mae"]):
        best_ridge_meta = candidate
if best_ridge_meta is not None:
    register_candidate("residual_ridge_meta", "meta_ridge", best_ridge_meta["oof_pred"], best_ridge_meta["test_pred"], weights=best_ridge_meta["weights"], params=best_ridge_meta["params"], eligible_for_selection=True)
    advanced_diagnostics["candidate_search"]["residual_ridge_meta"] = {"grid_results": ridge_meta_rows, "selected": best_ridge_meta["params"]}
    print("Selected residual_ridge_meta by OOF:", best_ridge_meta["params"])

elasticnet_rows = []
best_elasticnet = None
try:
    for alpha in [0.0001, 0.001, 0.01]:
        for l1_ratio in [0.1, 0.5]:
            model = ElasticNet(alpha=float(alpha), l1_ratio=float(l1_ratio), fit_intercept=True, max_iter=3000, tol=1e-3, random_state=42)
            model.fit(X_val_meta, y_res_val)
            oof_pred = clip_rating(y_val_base + model.predict(X_val_meta))
            test_pred = clip_rating(y_base + model.predict(X_test_meta))
            oof_res = evaluate_predictions(y_val_true, oof_pred, f"elasticnet_residual_meta_alpha_{alpha}_l1_{l1_ratio}")
            row = {"alpha": float(alpha), "l1_ratio": float(l1_ratio), "oof_rmse": float(oof_res["rmse"]), "oof_mae": float(oof_res["mae"]), "intercept": float(model.intercept_), "coef_by_feature": {name: float(coef) for name, coef in zip(meta_feature_names, model.coef_)}}
            elasticnet_rows.append(row)
            candidate = {"params": {"alpha": float(alpha), "l1_ratio": float(l1_ratio), "fit_intercept": True, "feature_names": meta_feature_names}, "weights": {"intercept": float(model.intercept_), "coef_by_feature": {name: float(coef) for name, coef in zip(meta_feature_names, model.coef_)}}, "oof_pred": oof_pred.copy(), "test_pred": test_pred.copy(), "metrics": oof_res}
            if best_elasticnet is None or (oof_res["rmse"], oof_res["mae"]) < (best_elasticnet["metrics"]["rmse"], best_elasticnet["metrics"]["mae"]):
                best_elasticnet = candidate
    if best_elasticnet is not None:
        register_candidate("elasticnet_residual_meta", "meta_elasticnet", best_elasticnet["oof_pred"], best_elasticnet["test_pred"], weights=best_elasticnet["weights"], params=best_elasticnet["params"], eligible_for_selection=True)
        advanced_diagnostics["candidate_search"]["elasticnet_residual_meta"] = {"grid_results": elasticnet_rows, "selected": best_elasticnet["params"]}
        print("Selected elasticnet_residual_meta by OOF:", best_elasticnet["params"])
except Exception as exc:
    advanced_diagnostics["skipped_candidates"]["elasticnet_residual_meta"] = str(exc)
    print("Skipped elasticnet_residual_meta:", exc)

hgbr_rows = []
best_hgbr = None
try:
    rng = np.random.RandomState(42)
    if len(X_val_meta) > 500000:
        sample_size = min(300000, len(X_val_meta))
        sample_idx = rng.choice(len(X_val_meta), size=sample_size, replace=False)
    else:
        sample_idx = np.arange(len(X_val_meta))
    X_train_hg = X_val_meta[sample_idx]
    y_train_hg = y_res_val[sample_idx]
    for max_iter in [100, 200]:
        for learning_rate in [0.03, 0.05]:
            for max_leaf_nodes in [15, 31]:
                for l2_regularization in [0.0, 0.1]:
                    model = HistGradientBoostingRegressor(loss="squared_error", max_iter=int(max_iter), learning_rate=float(learning_rate), max_leaf_nodes=int(max_leaf_nodes), l2_regularization=float(l2_regularization), random_state=42)
                    model.fit(X_train_hg, y_train_hg)
                    oof_pred = clip_rating(y_val_base + model.predict(X_val_meta))
                    test_pred = clip_rating(y_base + model.predict(X_test_meta))
                    oof_res = evaluate_predictions(y_val_true, oof_pred, f"hist_gradient_boosting_residual_{max_iter}_{learning_rate}_{max_leaf_nodes}_{l2_regularization}")
                    row = {"max_iter": int(max_iter), "learning_rate": float(learning_rate), "max_leaf_nodes": int(max_leaf_nodes), "l2_regularization": float(l2_regularization), "train_rows_used": int(len(sample_idx)), "oof_rmse": float(oof_res["rmse"]), "oof_mae": float(oof_res["mae"])}
                    hgbr_rows.append(row)
                    candidate = {"params": {"max_iter": int(max_iter), "learning_rate": float(learning_rate), "max_leaf_nodes": int(max_leaf_nodes), "l2_regularization": float(l2_regularization), "train_rows_used": int(len(sample_idx)), "feature_names": meta_feature_names}, "weights": {"model_type": "HistGradientBoostingRegressor"}, "oof_pred": oof_pred.copy(), "test_pred": test_pred.copy(), "metrics": oof_res}
                    if best_hgbr is None or (oof_res["rmse"], oof_res["mae"]) < (best_hgbr["metrics"]["rmse"], best_hgbr["metrics"]["mae"]):
                        best_hgbr = candidate
    if best_hgbr is not None:
        register_candidate("hist_gradient_boosting_residual", "meta_hist_gradient_boosting", best_hgbr["oof_pred"], best_hgbr["test_pred"], weights=best_hgbr["weights"], params=best_hgbr["params"], eligible_for_selection=True)
        advanced_diagnostics["candidate_search"]["hist_gradient_boosting_residual"] = {"grid_results": hgbr_rows, "selected": best_hgbr["params"]}
        print("Selected hist_gradient_boosting_residual by OOF:", best_hgbr["params"])
except Exception as exc:
    advanced_diagnostics["skipped_candidates"]["hist_gradient_boosting_residual"] = str(exc)
    print("Skipped hist_gradient_boosting_residual:", exc)

user_count_map = train.groupby(COL_USER).size()
item_count_map = train.groupby(COL_ITEM).size()
user_count_val = validation_index_oof[COL_USER].map(user_count_map).fillna(0).astype(np.int32).values
item_count_val = validation_index_oof[COL_ITEM].map(item_count_map).fillna(0).astype(np.int32).values
user_count_test = test_warm[COL_USER].map(user_count_map).fillna(0).astype(np.int32).values
item_count_test = test_warm[COL_ITEM].map(item_count_map).fillna(0).astype(np.int32).values
val_groups = build_group_labels(user_count_val, item_count_val)
test_groups = build_group_labels(user_count_test, item_count_test)

group_oof_pred, group_test_pred, group_summary = fit_group_aware_ridge_candidate(X_val_base_b_c_d, X_test_base_b_c_d, y_val_true, y_pred_ridge_oof, y_pred_ridge, val_groups, test_groups, min_rows=5000, alpha=1.0)
register_candidate("group_aware_ridge", "group_aware_ridge", group_oof_pred, group_test_pred, weights={"fallback_model": "ridge_positive_base_b_c_d"}, params={"alpha": 1.0, "positive": True, "fit_intercept": False, "feature_names": ["base", "b", "c", "d"], "min_group_rows": 5000, "group_summary": group_summary}, eligible_for_selection=True)
advanced_diagnostics["candidate_search"]["group_aware_ridge"] = {"feature_names": ["base", "b", "c", "d"], "min_group_rows": 5000, "group_summary": group_summary}
print("Built group_aware_ridge with groups:", sorted(set(val_groups.tolist())))

ensemble_extra_df = pd.DataFrame(compare_rows).sort_values(["oof_rmse", "oof_mae", "model"]).reset_index(drop=True)
display(ensemble_extra_df)


In [ ]:
# ============================================
# CELL 9 ? COMPARE AND SELECT BY OOF ONLY
# ============================================
compare_df = pd.DataFrame(compare_rows).sort_values(["eligible_for_selection", "oof_rmse", "oof_mae", "model"], ascending=[False, True, True, True]).reset_index(drop=True)
display(compare_df)

eligible_df = compare_df[compare_df["eligible_for_selection"]].sort_values(["oof_rmse", "oof_mae", "model"]).reset_index(drop=True)
assert len(eligible_df) > 0, "No eligible candidates were generated."

best_stage4_name = eligible_df.iloc[0]["model"]
best_stage4_candidate = candidate_store[best_stage4_name]
best_stage4_family = best_stage4_candidate["family"]
best_stage4_pred = clip_rating(best_stage4_candidate["test_pred"]).astype(np.float32)
best_stage4_oof_pred = clip_rating(best_stage4_candidate["oof_pred"]).astype(np.float32)
best_stage4_weights = best_stage4_candidate["weights"]
best_stage4_params = best_stage4_candidate["params"]
best_stage4_oof_metrics = evaluate_predictions(y_val_true, best_stage4_oof_pred, best_stage4_name)
best_stage4_test_metrics = evaluate_predictions(y_true, best_stage4_pred, best_stage4_name)

baseline_test_metrics = evaluate_predictions(y_true, y_base, "baseline")
baseline_test_per_star = evaluate_per_star(y_true, y_base)
best_test_per_star = evaluate_per_star(y_true, best_stage4_pred)

comparison_vs_current_ridge = None
if "ridge_positive_base_b_c_d" in candidate_store:
    ridge_candidate = candidate_store["ridge_positive_base_b_c_d"]
    ridge_oof_metrics = evaluate_predictions(y_val_true, ridge_candidate["oof_pred"], "ridge_positive_base_b_c_d")
    ridge_test_metrics = evaluate_predictions(y_true, ridge_candidate["test_pred"], "ridge_positive_base_b_c_d")
    comparison_vs_current_ridge = {
        "reference_model": "ridge_positive_base_b_c_d",
        "reference_family": ridge_candidate["family"],
        "oof_reference_metrics": ridge_oof_metrics,
        "test_reference_metrics": ridge_test_metrics,
        "oof_delta_rmse": float(best_stage4_oof_metrics["rmse"] - ridge_oof_metrics["rmse"]),
        "oof_delta_mae": float(best_stage4_oof_metrics["mae"] - ridge_oof_metrics["mae"]),
        "delta_rmse": float(best_stage4_test_metrics["rmse"] - ridge_test_metrics["rmse"]),
        "delta_mae": float(best_stage4_test_metrics["mae"] - ridge_test_metrics["mae"]),
    }

advanced_ensemble_improved = bool(best_stage4_family in ADVANCED_FAMILIES and comparison_vs_current_ridge is not None and (comparison_vs_current_ridge["delta_rmse"] < 0 or comparison_vs_current_ridge["delta_mae"] < 0))

selected_preview = {
    "selected_model": best_stage4_name,
    "selected_family": best_stage4_family,
    "selected_by": "oof_rmse_then_oof_mae",
    "oof_metrics": to_jsonable(best_stage4_oof_metrics),
    "test_metrics": to_jsonable(best_stage4_test_metrics),
    "weights": to_jsonable(best_stage4_weights),
    "params": to_jsonable(best_stage4_params),
    "note": "test_warm was used only for final evaluation, not selection",
}

print("Best OOF candidate:", best_stage4_name)
print(json.dumps(to_jsonable(selected_preview), indent=2))
print("Best test result after applying selected candidate:")
print(json.dumps(to_jsonable(best_stage4_test_metrics), indent=2))
if comparison_vs_current_ridge is not None:
    print("Comparison vs current ridge_positive_base_b_c_d:")
    print(json.dumps(to_jsonable(comparison_vs_current_ridge), indent=2))
    print("Advanced ensemble improved over current ridge_positive_base_b_c_d:", advanced_ensemble_improved)


In [ ]:
# ============================================
# CELL 10 ? SAVE STAGE 4 OUTPUTS
# ============================================
stage4_best_result = {
    "label": "best OOF-selected ensemble so far",
    "selected_model": best_stage4_name,
    "selected_family": best_stage4_family,
    "selected_by": "oof_rmse_then_oof_mae",
    "selection_source": "oof_validation",
    "oof_metrics": to_jsonable(best_stage4_oof_metrics),
    "test_metrics": to_jsonable(best_stage4_test_metrics),
    "weights": to_jsonable(best_stage4_weights),
    "params": to_jsonable(best_stage4_params),
    "note": "test_warm was used only for final evaluation, not selection",
}

stage4_weights = {
    "selected_model": best_stage4_name,
    "selected_family": best_stage4_family,
    "selected_by": "oof_rmse_then_oof_mae",
    "selected_weights": to_jsonable(best_stage4_weights),
    "selected_params": to_jsonable(best_stage4_params),
    "all_candidate_weights": {name: to_jsonable(info["weights"]) for name, info in candidate_store.items()},
    "all_candidate_params": {name: to_jsonable(info["params"]) for name, info in candidate_store.items()},
    "selection_source": "oof_validation",
}

stage4_advanced_diagnostics = {
    "selected_model": best_stage4_name,
    "selected_family": best_stage4_family,
    "best_oof_metrics": to_jsonable(best_stage4_oof_metrics),
    "best_test_metrics": to_jsonable(best_stage4_test_metrics),
    "baseline_test_metrics": to_jsonable(baseline_test_metrics),
    "best_test_per_star": to_jsonable(best_test_per_star),
    "baseline_test_per_star": to_jsonable(baseline_test_per_star),
    "comparison_vs_current_ridge_positive_base_b_c_d": to_jsonable(comparison_vs_current_ridge),
    "advanced_ensemble_improved_over_current_ridge": bool(advanced_ensemble_improved),
    "candidate_search": to_jsonable(advanced_diagnostics["candidate_search"]),
    "skipped_candidates": to_jsonable(advanced_diagnostics["skipped_candidates"]),
    "meta_feature_names": to_jsonable(advanced_diagnostics["meta_feature_names"]),
}

np.save(STAGE4_DIR / "stage4_best_preds.npy", best_stage4_pred)
compare_df.to_csv(STAGE4_DIR / "stage4_compare.csv", index=False)
with open(STAGE4_DIR / "stage4_best_result.json", "w") as f:
    json.dump(to_jsonable(stage4_best_result), f, indent=2)
with open(STAGE4_DIR / "stage4_weights.json", "w") as f:
    json.dump(to_jsonable(stage4_weights), f, indent=2)
with open(STAGE4_DIR / "stage4_advanced_diagnostics.json", "w") as f:
    json.dump(to_jsonable(stage4_advanced_diagnostics), f, indent=2)

print("Saved:")
print(STAGE4_DIR / "stage4_best_result.json")
print(STAGE4_DIR / "stage4_best_preds.npy")
print(STAGE4_DIR / "stage4_compare.csv")
print(STAGE4_DIR / "stage4_weights.json")
print(STAGE4_DIR / "stage4_advanced_diagnostics.json")


The Stage 4 notebook selects its final candidate only by strict OOF validation from Stage 3.

Advanced candidates now include residual ridge meta, elasticnet residual meta, HistGradientBoosting residual meta, and group-aware ridge. `test_warm` is applied only once after OOF selection to report final metrics.

If the selected candidate improves RMSE but worsens MAE versus `ridge_positive_base_b_c_d` or `model_b`, treat that as an explicit trade-off rather than a best-overall claim.
